# My Custom FAQ Chatbot (Semantic Search)

This notebook builds on the beginner RAG semantic search project and turns it into a
**reusable, interactive chatbot** with a **100-question knowledge base**.

New things added compared to the original beginner notebook:
- 100 FAQ question/answer pairs (10 categories, 10 each)
- A `Chatbot` class instead of loose functions
- A **confidence threshold** so it says "I don't know" instead of guessing when nothing matches well
- An **interactive chat loop** you can type into


In [ ]:
%pip install sentence-transformers scikit-learn numpy


## Imports

- `sentence-transformers`: converts text into vectors (embeddings).
- `cosine_similarity`: compares two vectors to score how similar their meaning is.
- `numpy`: array math, and finding the highest similarity score.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## Load the embedding model

The model converts sentences into numbers (vectors). Sentences with similar
meaning end up with similar vectors, even if they use different words.

In [ ]:
model = SentenceTransformer("all-MiniLM-L12-v2")
print("Model loaded")

## The FAQ knowledge base — 100 questions

This is your customized dataset: **100 question/answer pairs** across 10 topics
(Python basics, data structures, NumPy, Pandas, Machine Learning, Deep Learning,
NLP & embeddings, RAG & LLMs, Git/GitHub, and career/general AI advice).

Each item is a `(question, answer)` tuple, same format as the original notebook —
just scaled up from 5 to 100.

In [ ]:
faq_data = [
    ('What is Python?',
     'Python is a high-level, general-purpose programming language known for its simple, readable syntax. It is used for web development, automation, data science, and AI.'),
    ('What is a variable in Python?',
     'A variable is a name that stores a reference to a value in memory, such as a number, string, or object. Python variables do not need an explicit type declaration.'),
    ('What is a list in Python?',
     'A list is an ordered, mutable collection of items that can hold mixed data types, created using square brackets, e.g. [1, 2, 3].'),
    ('What is a dictionary in Python?',
     "A dictionary is a collection of key-value pairs where each key maps to a value, created using curly braces, e.g. {'name': 'Anmol'}."),
    ('What is a tuple in Python?',
     'A tuple is an ordered, immutable collection of items, created using parentheses, e.g. (1, 2, 3). Once created, its contents cannot be changed.'),
    ('What is a function in Python?',
     'A function is a reusable block of code defined with the def keyword that performs a specific task and can accept inputs and return outputs.'),
    ('What is a loop in Python?',
     'A loop repeats a block of code multiple times. Python has for loops for iterating over sequences and while loops for repeating while a condition is true.'),
    ('What is indentation in Python?',
     'Indentation is the whitespace at the start of a line used to define code blocks in Python, replacing the curly braces used in many other languages.'),
    ('What is a class in Python?',
     'A class is a blueprint for creating objects that bundle data (attributes) and behavior (methods) together, forming the basis of object-oriented programming.'),
    ('What is pip?',
     "Pip is Python's package installer, used to install and manage third-party libraries from the Python Package Index (PyPI)."),
    ('What is a stack?',
     'A stack is a linear data structure that follows Last-In-First-Out (LIFO) order, where elements are added and removed from the same end.'),
    ('What is a queue?',
     'A queue is a linear data structure that follows First-In-First-Out (FIFO) order, where elements are added at the back and removed from the front.'),
    ('What is a linked list?',
     'A linked list is a data structure made of nodes, where each node holds a value and a reference to the next node in the sequence.'),
    ('What is Big O notation?',
     'Big O notation describes how the runtime or memory usage of an algorithm grows as the input size increases, giving an upper bound on complexity.'),
    ('What is recursion?',
     'Recursion is a technique where a function calls itself to solve smaller instances of a problem until it reaches a base case.'),
    ('What is a binary search?',
     'Binary search is an efficient algorithm that finds an item in a sorted list by repeatedly dividing the search range in half.'),
    ('What is a hash table?',
     'A hash table is a data structure that maps keys to values using a hash function, allowing fast average-case lookup, insertion, and deletion.'),
    ('What is a tree data structure?',
     'A tree is a hierarchical data structure made of nodes, with a single root node and child nodes branching out, commonly used to represent hierarchical relationships.'),
    ('What is sorting?',
     'Sorting is the process of arranging elements of a collection in a particular order, typically ascending or descending, using algorithms like quicksort or mergesort.'),
    ('What is time complexity?',
     'Time complexity measures how the execution time of an algorithm scales relative to the size of its input.'),
    ('What is NumPy?',
     'NumPy is a Python library for numerical computing that provides fast, efficient array operations and mathematical functions.'),
    ('What is a NumPy array?',
     'A NumPy array (ndarray) is a fast, fixed-size, multidimensional container for elements of the same data type.'),
    ('Why use NumPy instead of Python lists?',
     'NumPy arrays are faster and more memory-efficient than Python lists because they are stored in contiguous memory and support vectorized operations.'),
    ('What is broadcasting in NumPy?',
     'Broadcasting is a NumPy feature that allows arithmetic operations on arrays of different shapes by automatically expanding the smaller array.'),
    ('What is the shape of a NumPy array?',
     'The shape of a NumPy array is a tuple describing the size of the array along each dimension, accessed via the .shape attribute.'),
    ('How do you create a NumPy array?',
     'You create a NumPy array using np.array() with a list or nested list, for example np.array([1, 2, 3]).'),
    ('What is np.argmax()?',
     'np.argmax() returns the index of the maximum value in a NumPy array, which is useful for finding the best match among similarity scores.'),
    ('What is a NumPy vector?',
     'A NumPy vector is a one-dimensional array of numbers, often used to represent embeddings or coordinates in machine learning.'),
    ('What is matrix multiplication in NumPy?',
     'Matrix multiplication in NumPy is performed using the @ operator or np.matmul(), combining rows and columns according to linear algebra rules.'),
    ('What is np.zeros() used for?',
     'np.zeros() creates a NumPy array filled with zeros of a specified shape, often used to initialize arrays before filling them with data.'),
    ('What is Pandas?',
     'Pandas is a Python library used for data analysis, providing powerful tools for working with structured, tabular data.'),
    ('What is a DataFrame?',
     'A DataFrame is a two-dimensional, labeled data structure in Pandas, similar to a table with rows and columns.'),
    ('What is a Pandas Series?',
     'A Series is a one-dimensional labeled array in Pandas, essentially a single column of a DataFrame.'),
    ('How do you read a CSV file in Pandas?',
     "You read a CSV file in Pandas using pd.read_csv('filename.csv'), which loads the data into a DataFrame."),
    ('What is the difference between loc and iloc in Pandas?',
     'loc selects data by label (row/column names), while iloc selects data by integer position in a DataFrame.'),
    ('How do you handle missing data in Pandas?',
     'Missing data in Pandas can be handled using methods like dropna() to remove missing values or fillna() to replace them.'),
    ('What is groupby in Pandas?',
     'groupby() splits a DataFrame into groups based on column values, allowing you to apply aggregate functions like sum or mean to each group.'),
    ('What is a Pandas index?',
     'An index in Pandas is the label used to identify rows in a DataFrame or Series, allowing fast lookup and alignment of data.'),
    ('How do you filter rows in Pandas?',
     "You filter rows in Pandas using boolean indexing, for example df[df['column'] > 5], which returns rows matching the condition."),
    ('What is merging in Pandas?',
     'Merging in Pandas combines two DataFrames based on a common column or index, similar to a SQL join, using pd.merge().'),
    ('What is Machine Learning?',
     'Machine Learning is a field of AI where computers learn patterns from data instead of being explicitly programmed with rules.'),
    ('What is supervised learning?',
     'Supervised learning is a type of machine learning where a model learns from labeled data, mapping inputs to known correct outputs.'),
    ('What is unsupervised learning?',
     'Unsupervised learning is a type of machine learning where a model finds patterns or groupings in data without labeled outputs.'),
    ('What is overfitting?',
     'Overfitting occurs when a model learns the training data too closely, including noise, and performs poorly on new, unseen data.'),
    ('What is a training set?',
     'A training set is the portion of data used to teach a machine learning model to recognize patterns before it is evaluated on new data.'),
    ('What is a feature in machine learning?',
     'A feature is an individual measurable input variable used by a machine learning model to make predictions.'),
    ('What is a label in machine learning?',
     'A label is the known correct output or target value associated with a training example in supervised learning.'),
    ('What is cross-validation?',
     'Cross-validation is a technique that splits data into multiple folds to train and test a model repeatedly, giving a more reliable estimate of its performance.'),
    ('What is a model in machine learning?',
     'A model is a mathematical representation learned from data that maps inputs to predictions or decisions.'),
    ('What is accuracy in machine learning?',
     'Accuracy is the proportion of correct predictions a model makes out of all predictions, commonly used to evaluate classification performance.'),
    ('What is Deep Learning?',
     'Deep Learning is a subset of machine learning that uses multi-layered neural networks to learn complex patterns from large amounts of data.'),
    ('What is a neural network?',
     'A neural network is a computing system made of layers of interconnected nodes, loosely inspired by the brain, used to learn patterns in data.'),
    ('What is a neuron in a neural network?',
     'A neuron is the basic unit of a neural network that receives inputs, applies weights and an activation function, and produces an output.'),
    ('What is an activation function?',
     'An activation function introduces non-linearity into a neural network, allowing it to learn complex patterns beyond simple linear relationships.'),
    ('What is backpropagation?',
     'Backpropagation is the algorithm used to train neural networks by calculating gradients of the error and updating weights to reduce that error.'),
    ('What is a Convolutional Neural Network?',
     'A Convolutional Neural Network (CNN) is a deep learning architecture specialized for processing grid-like data such as images, using convolutional filters.'),
    ('What is a Recurrent Neural Network?',
     'A Recurrent Neural Network (RNN) is a neural network architecture designed to process sequential data by maintaining a memory of previous inputs.'),
    ('What is a Transformer model?',
     'A Transformer is a neural network architecture that uses self-attention to process sequences in parallel, forming the basis of most modern language models.'),
    ('What is an epoch in deep learning?',
     'An epoch is one complete pass through the entire training dataset during the training of a machine learning model.'),
    ('What is a loss function?',
     "A loss function measures how far a model's predictions are from the true values, guiding the training process to minimize error."),
    ('What is Natural Language Processing?',
     'Natural Language Processing (NLP) is a field of AI focused on enabling computers to understand, interpret, and generate human language.'),
    ('What is a text embedding?',
     'A text embedding is a numerical vector representation of text that captures its meaning, allowing similar meanings to have similar vectors.'),
    ('What is sentence-transformers?',
     'sentence-transformers is a Python library that converts sentences into dense vector embeddings using pretrained transformer models.'),
    ('What is cosine similarity?',
     'Cosine similarity measures how similar two vectors are by calculating the cosine of the angle between them, with values closer to 1 meaning more similar.'),
    ('What is tokenization?',
     'Tokenization is the process of splitting text into smaller units, such as words or subwords, that a model can process.'),
    ('What is semantic search?',
     'Semantic search finds results based on the meaning of a query rather than exact keyword matching, typically using embeddings and similarity scores.'),
    ('What is a vector database?',
     'A vector database is a specialized database designed to store and efficiently search high-dimensional embedding vectors.'),
    ('What is the difference between keyword search and semantic search?',
     'Keyword search matches exact words in a query, while semantic search compares meaning using embeddings, so it can match related concepts phrased differently.'),
    ('What does all-MiniLM-L12-v2 do?',
     'all-MiniLM-L12-v2 is a pretrained sentence-transformer model that converts sentences into 384-dimensional embedding vectors capturing their meaning.'),
    ('Why do embeddings have a fixed dimension?',
     'Embeddings have a fixed dimension because the model architecture is designed to output vectors of a consistent size, enabling reliable comparison between them.'),
    ('What is RAG?',
     'RAG stands for Retrieval-Augmented Generation, a technique that combines information retrieval with text generation to produce more accurate, grounded answers.'),
    ('How does RAG work?',
     'RAG works by retrieving relevant documents or passages based on a query and then feeding them to a language model to generate a grounded response.'),
    ('What is a Large Language Model?',
     'A Large Language Model (LLM) is a deep learning model trained on massive amounts of text data to understand and generate human-like language.'),
    ('Why use RAG instead of just an LLM?',
     "RAG reduces hallucination and keeps answers up to date by grounding an LLM's response in retrieved, relevant external information rather than relying only on its training data."),
    ('What is a retriever in a RAG system?',
     'A retriever is the component of a RAG system that searches a knowledge base and returns the most relevant pieces of information for a given query.'),
    ('What is chunking in RAG?',
     'Chunking is the process of splitting large documents into smaller, manageable pieces so they can be embedded and retrieved effectively.'),
    ('What is a knowledge base in RAG?',
     'A knowledge base in RAG is the collection of documents or data that the retriever searches through to find relevant context for a query.'),
    ('What is hallucination in LLMs?',
     'Hallucination is when a language model generates information that sounds plausible but is factually incorrect or not grounded in real data.'),
    ('What is prompt engineering?',
     'Prompt engineering is the practice of designing effective inputs to guide a language model toward producing the desired output.'),
    ('What is context window in an LLM?',
     'The context window is the maximum amount of text an LLM can consider at once when generating a response, measured in tokens.'),
    ('What is Git?',
     'Git is a distributed version control system used to track changes in code and collaborate with others on software projects.'),
    ('What is GitHub?',
     'GitHub is a cloud-based platform for hosting Git repositories, enabling collaboration, code review, and project management.'),
    ('What is a Git commit?',
     'A commit is a saved snapshot of changes in a Git repository, recorded with a message describing what was changed.'),
    ('What is a Git branch?',
     'A branch is an independent line of development in Git, allowing you to work on features or fixes without affecting the main codebase.'),
    ('What is a pull request?',
     'A pull request is a request to merge changes from one branch into another, typically used for code review before merging on GitHub.'),
    ('What is git clone?',
     'git clone creates a local copy of a remote repository on your machine, including its full history.'),
    ('What is a merge conflict?',
     'A merge conflict occurs when Git cannot automatically combine changes from different branches because they modify the same lines of code.'),
    ('What is a .gitignore file?',
     'A .gitignore file specifies which files or folders Git should ignore and not track, such as temporary files or credentials.'),
    ('What is the difference between git push and git pull?',
     'git push uploads local commits to a remote repository, while git pull downloads and merges changes from the remote repository into your local copy.'),
    ('What is version control?',
     'Version control is a system that records changes to files over time, allowing you to track history, collaborate, and revert to previous versions.'),
    ('How do I start learning AI as a beginner?',
     'Start by learning Python fundamentals, then move into libraries like NumPy and Pandas, followed by core machine learning concepts and small hands-on projects.'),
    ('What skills are needed for a career in AI?',
     'Key skills include Python programming, statistics, linear algebra, machine learning fundamentals, and practical experience building and deploying projects.'),
    ('What is the difference between AI, ML, and Deep Learning?',
     'AI is the broad field of building intelligent systems, Machine Learning is a subset of AI focused on learning from data, and Deep Learning is a subset of ML using neural networks.'),
    ('What is a good first AI project for beginners?',
     'A good first project is a small semantic search or FAQ chatbot, since it introduces embeddings, similarity search, and practical Python coding in a manageable scope.'),
    ('What is Google Colab?',
     'Google Colab is a free, cloud-based Jupyter notebook environment that lets you write and run Python code, including GPU-accelerated machine learning code, in your browser.'),
    ('What is a Jupyter Notebook?',
     'A Jupyter Notebook is an interactive coding environment that combines code, output, and text in a single document, widely used for data science and prototyping.'),
    ('What is an API?',
     'An API (Application Programming Interface) is a set of rules that allows different software systems to communicate and exchange data with each other.'),
    ('What is open source software?',
     'Open source software is software whose source code is publicly available, allowing anyone to view, use, modify, and distribute it.'),
    ('How do I stay motivated while learning AI alone?',
     'Break learning into small, achievable projects, track your progress, and build things you can show or share, since visible progress sustains motivation better than passive study.'),
    ('What is the best way to practice coding skills?',
     'The best way to practice is by building small real projects consistently, since applying concepts hands-on cements understanding far more than reading alone.'),
]

print(f"Loaded {len(faq_data)} FAQ entries")

## Separate questions and answers

Same as before: keep questions and answers in two aligned lists so we can embed
the questions and still look up the matching answer by index.

In [ ]:
questions = [item[0] for item in faq_data]
answers = [item[1] for item in faq_data]

print(f"{len(questions)} questions ready to embed")
print(questions[:5], "...")

## Create embeddings for all 100 questions

Each question becomes a 384-number vector. With 100 questions, the resulting
shape will be `(100, 384)` — one row per question.

In [ ]:
question_embeddings = model.encode(questions, show_progress_bar=True)

print("Embedding shape:", question_embeddings.shape)

## Build a reusable `Chatbot` class

Instead of one throwaway function, this wraps everything into a class so it's
easy to reuse, extend, or plug into a Streamlit/Gradio app later.

Key upgrade over the beginner version: a **similarity threshold**. If the best
match score is too low, the bot admits it doesn't know rather than returning an
irrelevant FAQ.

In [ ]:
class FAQChatbot:
    """A simple semantic-search FAQ chatbot."""

    def __init__(self, model, questions, answers, threshold=0.45):
        self.model = model
        self.questions = questions
        self.answers = answers
        self.threshold = threshold
        self.question_embeddings = self.model.encode(self.questions)

    def ask(self, query, top_k=3, verbose=True):
        """Return the best-matching answer, or a fallback if confidence is low."""
        query_embedding = self.model.encode([query])
        similarity = cosine_similarity(query_embedding, self.question_embeddings)[0]

        # Top-k matches, sorted by score descending
        top_indices = np.argsort(similarity)[::-1][:top_k]
        best_index = top_indices[0]
        best_score = similarity[best_index]

        result = {
            "query": query,
            "matched_question": self.questions[best_index],
            "answer": self.answers[best_index],
            "score": round(float(best_score), 3),
            "confident": bool(best_score >= self.threshold),
            "top_matches": [
                (self.questions[i], round(float(similarity[i]), 3)) for i in top_indices
            ],
        }

        if verbose:
            print(f"You: {query}")
            if result["confident"]:
                print(f"Bot: {result['answer']}")
                print(f"   (matched: \"{result['matched_question']}\", score: {result['score']})")
            else:
                print("Bot: I'm not confident I have a good answer for that yet. "
                      "Could you rephrase, or ask about one of the topics I cover "
                      "(Python, ML, Deep Learning, NLP, RAG, Git, or AI careers)?")
                print(f"   (closest match: \"{result['matched_question']}\", score: {result['score']})")
            print()

        return result


# Create the chatbot instance
bot = FAQChatbot(model, questions, answers, threshold=0.45)
print("Chatbot ready with", len(questions), "FAQs loaded.")

## Test the chatbot

A few sample questions — including a paraphrase, an exact match, and an
off-topic question that should trigger the low-confidence fallback.

In [ ]:
bot.ask("Explain Pandas to me")

In [ ]:
bot.ask("What is RAG?")

In [ ]:
bot.ask("How do neural networks learn?")

In [ ]:
bot.ask("Difference between git push and pull")

In [ ]:
bot.ask("What is the capital of France?")  # off-topic -> should trigger fallback

## Chat interactively

Run this cell and type your own questions. Type `exit` or `quit` to stop.

> Note: This cell uses `input()`, so it only works in an interactive session
> (Jupyter, Colab, or a local terminal) — not when a notebook is run
> non-interactively end-to-end.

In [ ]:
print("FAQ Chatbot ready! Ask me about Python, ML, Deep Learning, NLP, RAG, Git, or AI careers.")
print("Type \'exit\' or \'quit\' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("exit", "quit"):
        print("Bot: Goodbye!")
        break
    bot.ask(user_input)

## Ideas to extend this further

- **Add more FAQs**: append more `(question, answer)` tuples to `faq_data` — no other code needs to change.
- **Persist embeddings**: save `question_embeddings` with `np.save()` so you don't recompute them every run.
- **Swap the UI**: wrap `bot.ask()` in a small [Gradio](https://gradio.app) or [Streamlit](https://streamlit.io) app for a shareable web chatbot.
- **Answer multiple FAQs at once**: use `result["top_matches"]` to show the top 3 candidate answers instead of just one.
- **Tune the threshold**: raise `threshold` (e.g. 0.55) for stricter matching, or lower it (e.g. 0.35) if it's rejecting valid questions too often.